# 01 · Thu thập dữ liệu

- **Người phụ trách:** Đỗ Khánh Duy
- **Đầu vào:** API danh sách tin và API chi tiết tin của Chợ Tốt.
- **Đầu ra chính:** `data/raw/cars.csv` và `data/raw/descriptions.csv`.
- **Đầu ra kiểm tra:** `data/raw/description_status.csv` và `data/raw/descriptions_report.json`.
- **Liên quan proposal:** mục 3 (Dữ liệu) và phương án bổ sung mô tả dài ở mục 5.

Notebook trình bày theo trình tự: **nguồn dữ liệu → cách crawler hoạt động → mô tả dữ liệu thô → cào bổ sung mô tả dài → kết luận**. Các bảng kết quả được tính trực tiếp từ file trong dự án.

**Cách sử dụng:** mở notebook trong thư mục dự án và chạy các cell từ trên xuống. Môi trường cần Python 3.10 trở lên, `pandas` và `ipykernel` (có trong `requirements.txt`). Hai tùy chọn chạy thử crawler mặc định là `False`; ở cấu hình này, **Run All chỉ đọc dữ liệu cục bộ**, không thu thập lại dữ liệu.

In [1]:
import ast
import json
import runpy
import subprocess
import sys
from pathlib import Path
from urllib.parse import urlencode

import pandas as pd
from IPython.display import HTML, Markdown, display

# Tìm gốc dự án dù kernel được mở từ gốc repo hoặc thư mục notebook.
ROOT = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "requirements.txt").is_file()
        and (folder / "01_crawler_and_ml" / "crawler").is_dir()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Hãy đặt notebook trong dự án và mở Jupyter từ gốc dự án "
        "hoặc thư mục 01_crawler_and_ml."
    )

RAW_DIR = ROOT / "data" / "raw"
RAW_PATH = RAW_DIR / "cars.csv"
DESCRIPTIONS_PATH = RAW_DIR / "descriptions.csv"
STATUS_PATH = RAW_DIR / "description_status.csv"
REPORT_PATH = RAW_DIR / "descriptions_report.json"
SCRAPER_PATH = ROOT / "01_crawler_and_ml" / "crawler" / "data_raw.py"
DESCRIPTION_SCRIPT = SCRAPER_PATH.with_name("descriptions.py")

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 90)
print("Đã thiết lập đường dẫn tương đối theo gốc dự án.")


def show_table(frame, formats=None):
    """Hiển thị đủ bảng, ẩn chỉ số dòng và giữ an toàn cho nội dung HTML."""
    formatters = {column: pattern.format for column, pattern in (formats or {}).items()}
    display(HTML(frame.to_html(index=False, escape=True, border=0, formatters=formatters)))

Đã thiết lập đường dẫn tương đối theo gốc dự án.


## 1. Nguồn dữ liệu

Dự án sử dụng tin rao bán ô tô đã qua sử dụng trên Chợ Tốt. Một tin có thể cung cấp giá niêm yết, hãng xe, dòng xe, năm sản xuất, số kilomet đã đi, các thuộc tính kỹ thuật, địa bàn và thời điểm đăng. Đây là dữ liệu do người đăng cung cấp; giá niêm yết chưa phải giá giao dịch thực tế.

Trong phương án triển khai hiện có, nhóm chọn API Chợ Tốt thay cho việc phân tích HTML của Bonbanh vì phản hồi JSON đã tách thông tin thành các trường. Cách tiếp cận này giảm phần xử lý giao diện trang web và thuận tiện khi xuất bảng dữ liệu. Cấu trúc API vẫn có thể thay đổi; các tham số và trường dưới đây được mô tả theo mã crawler đang có trong dự án.

| Nguồn | Địa chỉ dùng trong mã nguồn | Vai trò |
|---|---|---|
| Danh sách tin | `https://gateway.chotot.com/v1/public/ad-listing` | Thu thập thuộc tính của tin, gồm tiêu đề `subject` |
| Chi tiết tin | `https://gateway.chotot.com/v1/public/ad-listing/{list_id}` | Lấy nội dung `body`, lưu thành `description` |

Phạm vi truy vấn là toàn quốc, danh mục ô tô (`cg=2010`), tình trạng đã sử dụng (`condition_ad=1`). Crawler loại trùng theo mã tin `list_id`; nhiều tin khác mã vẫn có thể cùng rao một chiếc xe. Tập thu thập phục vụ phân tích tin đăng trên nền tảng, không mặc nhiên đại diện cho toàn bộ thị trường xe cũ Việt Nam.

## 2. Cách crawler hoạt động

Mã thu thập nằm tại [`crawler/data_raw.py`](crawler/data_raw.py). Luồng xử lý gồm: tạo truy vấn theo dải giá, đọc từng trang JSON, kiểm tra bản ghi, lưu vào SQLite, cập nhật tiến độ và xuất dữ liệu.

### 2.1. Endpoint, bộ lọc và phân trang

| Tham số | Giá trị trong crawler | Ý nghĩa sử dụng |
|---|---|---|
| `cg` | `2010` | Danh mục ô tô |
| `condition_ad` | `1` | Xe đã qua sử dụng |
| `st` | `s,k` | Giá trị bộ lọc được giữ nguyên từ crawler |
| `limit` | `50` | Số tin yêu cầu mỗi trang |
| `o` | `offset` | Vị trí bắt đầu trang tiếp theo |
| `price` | `low-high` | Khoảng giá truy vấn, đơn vị đồng |

Truy vấn không truyền bộ lọc tỉnh/thành. Sau mỗi trang, `offset` tăng theo số bản ghi thực sự đã duyệt; `INSERT OR IGNORE` giúp một `list_id` chỉ được lưu một lần.

### 2.2. Chia 11 dải giá

Crawler duyệt 11 dải giá tăng dần để giảm nguy cơ chạm ngưỡng 10.000 kết quả cho một truy vấn được ghi chú trong mã nguồn. Hai dải liền nhau không chồng lấn: cận trên gửi đi bằng cận dưới của dải kế tiếp trừ 1 đồng.

Cách chia này không tự bảo đảm mỗi dải có dưới 10.000 tin. Ngoài ra, crawler dừng khi đủ `--target`, nên các dải giá cao có thể chưa được duyệt hết. Đây là một giới hạn cần lưu ý khi phân tích phân bố giá.

In [ ]:
# Đọc cấu hình từ mã nguồn; run_path không gọi khối if __name__ == "__main__".
scraper_config = runpy.run_path(str(SCRAPER_PATH))
ENDPOINT = scraper_config["ENDPOINT"]
EXPECTED_COLUMNS = scraper_config["FIELDS"] + ["source_url", "fetched_at"]

# Lấy chính danh sách bounds trong data_raw.py để bảng không lệch cấu hình.
scraper_tree = ast.parse(SCRAPER_PATH.read_text(encoding="utf-8"))
bounds_node = next(
    node.value
    for node in ast.walk(scraper_tree)
    if isinstance(node, ast.Assign)
    and any(isinstance(target, ast.Name) and target.id == "bounds" for target in node.targets)
)
bounds = ast.literal_eval(bounds_node)
price_bands = pd.DataFrame(
    [
        {
            "Dải": index,
            "Giá từ (đồng)": low,
            "Giá đến (đồng)": high - 1,
            "Tham số price": f"{low}-{high - 1}",
        }
        for index, (low, high) in enumerate(zip(bounds, bounds[1:]), start=1)
    ]
)
show_table(price_bands, {"Giá từ (đồng)": "{:,.0f}", "Giá đến (đồng)": "{:,.0f}"})

example_params = {
    "cg": 2010, "condition_ad": 1, "st": "s,k",
    "limit": 50, "o": 0, "price": price_bands.loc[0, "Tham số price"],
}
print("URL minh họa (chưa gửi yêu cầu):")
print(ENDPOINT + "?" + urlencode(example_params))
print(f"Số trường xuất CSV: {len(EXPECTED_COLUMNS)}")

### 2.3. Lưu tiến độ và xử lý lỗi

- **Kiểm tra bản ghi:** danh mục phải là `2010`, tình trạng phải là `1`, có `list_id` và giá nằm trong dải đang truy vấn. Crawler dừng nếu dữ liệu trả về không thỏa điều kiện.
- **Checkpoint SQLite:** bảng `cars` lưu JSON với khóa chính `list_id`; bảng `progress` lưu dải giá, `offset` và cờ hoàn tất. Dùng lại cùng thư mục `--out` để tiếp tục từ checkpoint.
- **Giới hạn tốc độ:** mặc định chờ 1,5 giây giữa hai trang; tham số `--delay` tối thiểu là 1 giây. Khi gặp HTTP 429, crawler đọc `Retry-After` dạng số giây và chờ ít nhất 60 giây.
- **Thử lại:** tối đa 5 lần gửi cho một trang. Lỗi máy chủ từ HTTP 500 trở lên và lỗi mạng/thời gian chờ được xử lý theo logic thử lại trong `fetch()`. Năm trang liên tiếp không thêm mã tin mới sẽ làm crawler dừng.
- **Xuất kết quả:** khối `finally` xuất `cars.csv`, `cars.jsonl` và `report.json` từ SQLite khi luồng chạy thoát theo cơ chế xử lý của Python. Danh sách ảnh được mã hóa thành chuỗi JSON trong CSV; `source_url` và `fetched_at` ghi nguồn truy vấn và thời điểm thu thập (UTC).

SQLite và JSONL phục vụ thu thập/chạy tiếp; CSV là đầu vào bàn giao cho các notebook sau. Thư mục dự án hiện có CSV; không nên suy ra đã có checkpoint chỉ từ sự tồn tại của CSV.

### 2.4. Chạy thử với số tin nhỏ

Cell dưới cho phép chạy thử tối đa 100 tin trong thư mục riêng `data/sample_collection`. Chuyển `RUN_CRAWLER_DEMO` thành `True` khi muốn gửi yêu cầu mạng.

Để thu thập bộ chính, chạy lệnh sau **từ gốc dự án**:

```bash
python 01_crawler_and_ml/crawler/data_raw.py --target 15000 --delay 1.5 --out ../../data/raw
```

Trong `data_raw.py`, `--out` tương đối được tính từ thư mục chứa script; vì vậy cần `../../data/raw` để ghi đúng thư mục dữ liệu chung. Tham số `--target` là tổng số mã tin mong muốn trong checkpoint, không phải số tin mới cần thêm.

In [3]:
RUN_CRAWLER_DEMO = False
DEMO_TARGET = 100

if RUN_CRAWLER_DEMO:
    command = [
        sys.executable, str(SCRAPER_PATH),
        "--target", str(DEMO_TARGET), "--delay", "1.5",
        "--out", str(ROOT / "data" / "sample_collection"),
    ]
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print("Đã bỏ qua lượt thu thập thử; tiếp tục đọc data/raw/cars.csv.")

## 3. Mô tả dữ liệu thô

Phần này đọc `data/raw/cars.csv`, kiểm tra kích thước, khóa định danh và trình bày từ điển dữ liệu. Các phép kiểm tra chỉ báo cáo chất lượng; việc giải mã, chuẩn hóa và xử lý ngoại lai thuộc notebook 02.

### 3.1. Kích thước và mẫu dữ liệu

Giữ `list_id` và `ad_id` ở kiểu chuỗi để sử dụng như mã định danh. Các kiểu dữ liệu còn lại do pandas suy luận khi đọc CSV.

In [4]:
if not RAW_PATH.is_file():
    raise FileNotFoundError(
        "Thiếu data/raw/cars.csv. Hãy đặt dữ liệu vào đúng thư mục "
        "hoặc chạy crawler theo mục 2.4."
    )

raw = pd.read_csv(
    RAW_PATH, encoding="utf-8-sig", low_memory=False,
    dtype={"list_id": "string", "ad_id": "string"},
)
if raw.empty:
    raise ValueError("cars.csv chưa có bản ghi để mô tả.")
if list(raw.columns) != EXPECTED_COLUMNS:
    raise ValueError("Các cột trong cars.csv không khớp thứ tự trường xuất của data_raw.py.")

n_rows, n_columns = raw.shape
raw_ids = set(raw["list_id"].dropna())
fetched_times = pd.to_datetime(raw["fetched_at"], errors="coerce", utc=True)

overview = pd.DataFrame(
    [
        ("Số dòng", f"{n_rows:,}"),
        ("Số cột", str(n_columns)),
        ("Số mã tin duy nhất", f"{raw['list_id'].nunique():,}"),
        ("Số dòng thiếu mã tin", str(raw["list_id"].isna().sum())),
        ("Số dòng trùng mã tin (sau dòng đầu)", str(raw["list_id"].duplicated().sum())),
        ("Thu thập sớm nhất (UTC)", str(fetched_times.min())),
        ("Thu thập muộn nhất (UTC)", str(fetched_times.max())),
    ],
    columns=["Chỉ tiêu", "Giá trị"],
)
show_table(overview)
display(raw[
    ["list_id", "subject", "price", "carbrand_name", "carmodel_name",
     "mfdate", "mileage_v2", "region_name"]
].head())

### 3.2. Từ điển 35 trường dữ liệu

Bảng dưới giữ đúng thứ tự cột trong CSV, gồm **tên trường, kiểu dữ liệu khi đọc, ý nghĩa, số giá trị thiếu và tỷ lệ thiếu**. Tỷ lệ thiếu tính trên toàn bộ số dòng; giá trị rỗng hoặc chỉ có khoảng trắng cũng được tính là thiếu. Giá trị `0` không tự động được coi là thiếu.

Kiểu `float64` ở một cột mã số có thể xuất hiện vì cột có giá trị thiếu. Kiểu lưu trữ không quyết định ý nghĩa thống kê của trường: các mã phân loại vẫn cần xử lý như biến phân loại.

In [5]:
FIELD_MEANINGS = {
    "list_id": "Mã tin đăng; khóa nối với mô tả dài và khóa chống trùng của crawler.",
    "ad_id": "Mã quảng cáo do API trả về; giữ như mã định danh.",
    "subject": "Tiêu đề tin do người đăng nhập.",
    "price": "Giá niêm yết dạng số, đơn vị đồng Việt Nam.",
    "price_string": "Giá ở dạng chuỗi hiển thị, có ký hiệu tiền tệ.",
    "category": "Mã danh mục; crawler yêu cầu 2010 (ô tô).",
    "category_name": "Tên danh mục do API trả về.",
    "condition_ad": "Mã tình trạng; crawler yêu cầu 1 (đã sử dụng).",
    "condition_ad_name": "Tên tình trạng xe do API trả về.",
    "carbrand": "Mã hãng xe.",
    "carbrand_name": "Tên hãng xe.",
    "carmodel": "Mã dòng xe.",
    "carmodel_name": "Tên dòng xe.",
    "mfdate": "Năm sản xuất được khai báo trong tin.",
    "mileage_v2": "Số kilomet xe đã đi theo thông tin người đăng.",
    "gearbox": "Mã loại hộp số; giải mã ở notebook 02.",
    "fuel": "Mã loại nhiên liệu; giải mã ở notebook 02.",
    "carseats": "Mã nhóm số chỗ ngồi; cần bảng ánh xạ, không dùng trực tiếp như số chỗ.",
    "cartype": "Mã kiểu dáng xe; giải mã ở notebook 02.",
    "carcolor": "Mã màu xe; giải mã ở notebook 02.",
    "carorigin": "Mã xuất xứ xe; giải mã ở notebook 02.",
    "region_v2": "Mã địa bàn cấp tỉnh/thành theo trường region_v2 của API.",
    "region_name": "Tên tỉnh/thành theo trường region_name.",
    "region_name_v3": "Tên địa bàn theo trường region_name_v3; cần đối chiếu khi chuẩn hóa.",
    "area_name": "Tên quận/huyện hoặc khu vực do API trả về.",
    "ward_name": "Tên phường/xã do API trả về.",
    "list_time": "Mốc thời gian đăng/hiển thị theo API; giá trị số dạng Unix mili giây.",
    "orig_list_time": "Mốc thời gian đăng gốc theo API; cần kiểm tra đơn vị khi có dữ liệu.",
    "image": "URL ảnh đại diện của tin.",
    "images": "Danh sách URL ảnh được lưu thành chuỗi JSON trong CSV.",
    "number_of_images": "Số ảnh do API trả về.",
    "type": "Mã loại tin do API trả về; giữ nguyên mã, chưa diễn giải nhóm.",
    "status": "Trạng thái tin tại thời điểm lấy danh sách.",
    "source_url": "URL truy vấn danh sách đã trả về bản ghi.",
    "fetched_at": "Thời điểm crawler lấy bản ghi, chuỗi ISO 8601 theo UTC.",
}
if set(FIELD_MEANINGS) != set(raw.columns):
    raise ValueError("Từ điển dữ liệu chưa mô tả đủ các trường của CSV.")

blank_cells = raw.astype("string").apply(
    lambda column: column.str.strip().eq("").fillna(False)
)
missing_cells = raw.isna() | blank_cells
missing_counts = missing_cells.sum()
field_dictionary = pd.DataFrame({
    "STT": range(1, n_columns + 1),
    "Tên trường": raw.columns,
    "Kiểu pandas": [str(raw[column].dtype) for column in raw.columns],
    "Ý nghĩa": [FIELD_MEANINGS[column] for column in raw.columns],
    "Số thiếu": missing_counts.reindex(raw.columns).to_numpy(),
    "Tỷ lệ thiếu (%)": (missing_counts / n_rows * 100).reindex(raw.columns).to_numpy(),
})
show_table(field_dictionary, {"Tỷ lệ thiếu (%)": "{:.2f}"})

### 3.3. Kiểm tra dữ liệu trước khi bàn giao

Các cột `gearbox`, `fuel`, `cartype`, `carcolor`, `carorigin` và `carseats` đang là **mã phân loại**, chưa được giải mã ở notebook này. `carseats` cần được đối chiếu bảng mã trước khi diễn giải thành số chỗ thực tế.

Giá, năm sản xuất và kilomet cần được rà soát ở notebook 02. Một giá trị không thiếu vẫn có thể không hợp lý. Trường hợp một chiếc xe có nhiều tin khác `list_id` cũng cần tiêu chí riêng để nhận diện.

In [6]:
quality_checks = pd.DataFrame([
    ("Mã tin bị thiếu hoặc rỗng", int(missing_cells["list_id"].sum())),
    ("Mã tin trùng (sau lần xuất hiện đầu)", int(raw["list_id"].duplicated().sum())),
    ("Danh mục khác ô tô hoặc bị thiếu", int((raw["category"].ne(2010) | raw["category"].isna()).sum())),
    ("Tình trạng khác đã sử dụng hoặc bị thiếu", int((raw["condition_ad"].ne(1) | raw["condition_ad"].isna()).sum())),
    ("Giá thiếu hoặc không đọc được thành số", int(pd.to_numeric(raw["price"], errors="coerce").isna().sum())),
    ("Giá bằng 0 hoặc âm", int(pd.to_numeric(raw["price"], errors="coerce").le(0).sum())),
    ("Thời điểm thu thập thiếu hoặc không đọc được", int(fetched_times.isna().sum())),
], columns=["Nội dung kiểm tra", "Số dòng"])
show_table(quality_checks)

missing_summary = field_dictionary.loc[
    field_dictionary["Số thiếu"].gt(0),
    ["Tên trường", "Số thiếu", "Tỷ lệ thiếu (%)"],
].sort_values("Tỷ lệ thiếu (%)", ascending=False)
print(f"Có {len(missing_summary)}/{n_columns} cột chứa giá trị thiếu.")
show_table(missing_summary, {"Tỷ lệ thiếu (%)": "{:.2f}"})

coded_columns = ["gearbox", "fuel", "cartype", "carcolor", "carorigin", "carseats"]
coded_summary = pd.DataFrame({
    "Trường mã hóa": coded_columns,
    "Số mã khác nhau (không tính thiếu)": [raw[c].nunique() for c in coded_columns],
})
show_table(coded_summary)

## 4. Cào bổ sung mô tả dài

### 4.1. Mục đích và cách lấy dữ liệu

`cars.csv` có tiêu đề `subject` nhưng không có nội dung mô tả dài. Script [`crawler/descriptions.py`](crawler/descriptions.py) đọc danh sách `list_id` từ bộ dữ liệu gốc, gọi API chi tiết, kiểm tra mã tin và trạng thái tin, rồi lấy trường `body` để lưu thành `description`.

Crawler chỉ lưu mô tả không rỗng của tin có trạng thái `active`. Mô tả thiếu không được thay bằng tiêu đề trong file gốc. Các đầu ra gồm:

| File | Nội dung |
|---|---|
| `descriptions.csv` | Hai cột `list_id,description`; chỉ các tin lấy mô tả thành công |
| `description_status.csv` | Trạng thái kiểm tra của mọi mã tin gốc, kể cả `pending` |
| `descriptions_report.json` | Tổng số tin, độ phủ, thời gian, cấu hình và lý do dừng |
| `descriptions.sqlite3` | Checkpoint để tiếp tục lượt thu thập, được tạo khi chạy crawler |

Script lưu từng kết quả vào checkpoint, xuất CSV định kỳ và khi kết thúc. Khi chạy lại với cùng checkpoint, các trạng thái `ok`, `removed`, `empty`, `unavailable` được bỏ qua; các trạng thái lỗi được thử lại. Tùy chọn `--refresh` cho phép kiểm tra lại các trạng thái đã kết thúc. Chỉ có CSV không thay thế cho checkpoint SQLite khi muốn chạy tiếp.

### 4.2. Chạy thử và thu thập đầy đủ

Cell dưới chạy thử tối đa 20 mã tin vào 

Lệnh đầy đủ không đặt `--limit` hay `--max-seconds`. Khoảng cách tối thiểu giữa hai lần bắt đầu yêu cầu được áp dụng chung cho mọi worker. HTTP 429 làm các worker cùng chờ theo `Retry-After`, tối thiểu 60 giây. Crawler cũng dừng nếu gặp 10 kết quả lỗi liên tiếp; lý do dừng được lưu trong báo cáo.

In [1]:
RUN_DESCRIPTION_DEMO = False

if RUN_DESCRIPTION_DEMO:
    command = [
        sys.executable, str(DESCRIPTION_SCRIPT),
        "--input", str(RAW_PATH),
        "--out", str(ROOT / "data" / "sample_descriptions"),
        "--limit", "20", "--workers", "1", "--delay", "1",
        "--timeout", "20", "--max-attempts", "3",
    ]
    result = subprocess.run(command, cwd=ROOT, check=False)
    if result.returncode not in (0, 2):
        raise RuntimeError(f"Crawler dừng với mã lỗi {result.returncode}.")
    print(
        "Đã kết thúc lượt thử. Hãy đọc báo cáo trong data/sample_descriptions; "
        "mã thoát 2 có thể xuất hiện vì các mã tin ngoài giới hạn thử vẫn pending."
    )
else:
    print("Đã bỏ qua lượt cào mô tả thử; đọc kết quả bàn giao trong data/raw.")

### 4.3. Kiểm tra đầu ra và thống kê trạng thái

Quy ước `removed` chỉ áp dụng khi API chi tiết trả HTTP **404 hoặc 410** tại thời điểm kiểm tra. Đây là số tin không còn truy cập được theo tiêu chí của đợt cào, không xác định nguyên nhân hoặc ai đã gỡ tin. Tin `unavailable`, lỗi mạng và mã chưa kiểm tra không được cộng vào số tin đã gỡ.

Cell sau kiểm tra khóa nối, cấu trúc hai file CSV, nội dung mô tả và sự khớp nhau giữa CSV với báo cáo JSON trước khi tính kết quả. Nếu thiếu file, cần lấy lại đầu ra hoặc chạy crawler theo mục 4.2.

In [6]:
required_paths = [DESCRIPTIONS_PATH, STATUS_PATH, REPORT_PATH]
missing_paths = [str(path.relative_to(ROOT)) for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError("Thiếu đầu ra mô tả: " + ", ".join(missing_paths))

descriptions = pd.read_csv(
    DESCRIPTIONS_PATH, encoding="utf-8-sig", dtype="string", keep_default_na=False
)
status = pd.read_csv(
    STATUS_PATH, encoding="utf-8-sig", dtype="string", keep_default_na=False
)
report = json.loads(REPORT_PATH.read_text(encoding="utf-8-sig"))

status_labels = {
    "ok": "Có mô tả dài",
    "removed": "Không truy cập được chi tiết (HTTP 404/410)",
    "empty": "Tin còn hoạt động nhưng mô tả rỗng",
    "unavailable": "Tin không hoạt động, chưa rõ nguyên nhân",
    "http_error": "Lỗi HTTP khác",
    "network_error": "Lỗi mạng hoặc hết thời gian chờ",
    "invalid_response": "Phản hồi không hợp lệ",
    "rate_limited": "Bị giới hạn tốc độ",
    "pending": "Chưa kiểm tra",
}
status_order = list(status_labels)
expected_status_columns = {
    "list_id", "status", "http_status", "attempts", "checked_at", "source_url", "error"
}

assert list(descriptions.columns) == ["list_id", "description"], "Sai cấu trúc descriptions.csv."
assert set(status.columns) == expected_status_columns, "Sai cấu trúc description_status.csv."
assert not missing_cells["list_id"].any() and raw["list_id"].is_unique, "Mã tin gốc thiếu hoặc trùng."
assert status["list_id"].is_unique and descriptions["list_id"].is_unique, "Mã tin đầu ra bị trùng."
assert set(status["list_id"]) == raw_ids, "Nhật ký phải phủ đúng mọi mã tin gốc."
assert set(status["status"]) <= set(status_order), "Có trạng thái chưa được định nghĩa."
assert descriptions["description"].str.strip().ne("").all(), "Mô tả thành công bị rỗng."

ok_ids = set(status.loc[status["status"].eq("ok"), "list_id"])
assert set(descriptions["list_id"]) == ok_ids, "Mô tả không khớp các mã tin có trạng thái ok."
removed_rows = status.loc[status["status"].eq("removed")]
assert removed_rows["http_status"].isin(["404", "410"]).all(), "Trạng thái removed thiếu bằng chứng HTTP."

counts = status["status"].value_counts().reindex(status_order, fill_value=0).astype(int)
total = len(raw_ids)
checked = total - int(counts["pending"])
error_states = ["http_error", "network_error", "invalid_response", "rate_limited"]
error_count = int(counts.loc[error_states].sum())
unresolved = int(counts["pending"]) + error_count
coverage = 100 * len(descriptions) / total

assert report["total_ids"] == total and report["checked_ids"] == checked, "Tổng số trong JSON lệch CSV."
assert all(report["counts"].get(key, 0) == int(counts[key]) for key in status_order), "Số trạng thái trong JSON lệch CSV."
assert abs(report["description_coverage"] - coverage / 100) < 1e-9, "Độ phủ trong JSON lệch CSV."
assert report["unresolved_ids"] == unresolved, "Số tin chưa giải quyết trong JSON lệch CSV."

print("Đã kiểm tra: cấu trúc, mã tin, mô tả, bằng chứng HTTP và số liệu CSV/JSON đều khớp.")

NameError: name 'missing_cells' is not defined

In [9]:
status_summary = pd.DataFrame({
    "Trạng thái": status_order,
    "Ý nghĩa": [status_labels[value] for value in status_order],
    "Số tin": counts.to_numpy(),
    "Tỷ lệ trên toàn bộ tin (%)": counts.to_numpy() / total * 100,
})
show_table(status_summary, {"Số tin": "{:,}", "Tỷ lệ trên toàn bộ tin (%)": "{:.2f}"})

checked_times = pd.to_datetime(status["checked_at"], errors="coerce", utc=True)
run_info = report.get("run", {})
run_summary = pd.DataFrame([
    ("Số mã tin đã kiểm tra", f"{checked:,}/{total:,}"),
    ("Số mô tả dài", f"{len(descriptions):,}"),
    ("Độ phủ mô tả trên bộ gốc", f"{coverage:.2f}%"),
    ("Tin không truy cập được (HTTP 404/410)", f"{int(counts['removed']):,}"),
    ("Tin chưa có kết quả chắc chắn", f"{unresolved:,}"),
    ("Kiểm tra sớm nhất (UTC)", str(checked_times.min())),
    ("Kiểm tra muộn nhất (UTC)", str(checked_times.max())),
    ("Số worker của lượt được lưu", str(run_info.get("workers", "Không ghi nhận"))),
    ("Khoảng cách yêu cầu (giây)", str(run_info.get("delay", "Không ghi nhận"))),
    ("Lý do dừng", str(run_info.get("stop_reason", "Không ghi nhận"))),
], columns=["Chỉ tiêu", "Kết quả"])
show_table(run_summary)

if unresolved:
    print(
        f"Còn {unresolved:,} mã tin chưa có kết quả chắc chắn: "
        f"{int(counts['pending']):,} chưa kiểm tra và {error_count:,} gặp lỗi."
    )
else:
    print("Tất cả mã tin đã có kết quả kiểm tra trong bộ file bàn giao.")

### 4.4. Sử dụng mô tả trong notebook 04

Khi phân tích văn bản, nối trái theo `list_id` để giữ các tin thiếu mô tả. Ví dụ dưới dùng bộ dữ liệu gốc để minh họa; notebook 04 cần tính lại độ phủ trên bộ đã làm sạch thực tế dùng cho NLP.

Nếu dùng `subject` khi thiếu mô tả, tạo biến riêng và ghi rõ nguồn văn bản. Không ghi tiêu đề thay thế vào `descriptions.csv`. Mô tả được thu thập sau dữ liệu danh sách nên nội dung và trạng thái tin có thể thay đổi. Nhóm tin còn lấy được mô tả cũng có thể khác nhóm đã không còn truy cập được.

In [10]:
# Minh họa nối dữ liệu trong bộ nhớ; không ghi đè các file gốc.
text_data = raw[["list_id", "subject"]].merge(
    descriptions, on="list_id", how="left", validate="one_to_one"
)
assert len(text_data) == len(raw), "Phép nối làm thay đổi số dòng."
text_data["has_description"] = text_data["description"].fillna("").str.strip().ne("")
text_data["text_source"] = text_data["has_description"].map(
    {True: "description", False: "subject"}
)
text_data["text_for_nlp"] = text_data["description"].where(
    text_data["has_description"], text_data["subject"]
)

text_source_summary = (
    text_data["text_source"].value_counts()
    .reindex(["description", "subject"], fill_value=0)
    .rename_axis("Nguồn văn bản").reset_index(name="Số tin")
)
show_table(text_source_summary)
print(f"Giữ đủ {len(text_data):,} dòng sau phép nối trái theo list_id.")

## 5. Kết luận

Notebook đã trình bày nguồn tin Chợ Tốt, cơ chế thu thập theo dải giá, cấu trúc dữ liệu thô và bước bổ sung mô tả dài. Bảng bàn giao dưới đây lấy số lượng từ các file đã được đọc và kiểm tra trong lần chạy notebook.

In [11]:
handoff = pd.DataFrame([
    {
        "File": "data/raw/cars.csv",
        "Kích thước": f"{len(raw):,} dòng × {len(raw.columns)} cột",
        "Nơi sử dụng": "Notebook 02: làm sạch, chuẩn hóa dữ liệu",
    },
    {
        "File": "data/raw/descriptions.csv",
        "Kích thước": f"{len(descriptions):,} dòng × {len(descriptions.columns)} cột",
        "Nơi sử dụng": "Notebook 04: nối theo list_id để phân tích văn bản",
    },
    {
        "File": "data/raw/description_status.csv",
        "Kích thước": f"{len(status):,} dòng × {len(status.columns)} cột",
        "Nơi sử dụng": "Đối chiếu trạng thái của từng mã tin",
    },
    {
        "File": "data/raw/descriptions_report.json",
        "Kích thước": "1 báo cáo tổng hợp",
        "Nơi sử dụng": "Đối chiếu phạm vi, thời điểm và cấu hình thu thập",
    },
])
show_table(handoff)
display(Markdown(
    f"**Kết quả từ dữ liệu hiện có:** {len(raw):,} tin với {len(raw.columns)} trường; "
    f"{len(descriptions):,} tin có mô tả dài (**{coverage:.2f}%**). "
    f"Có {int(counts['removed']):,} tin không truy cập được chi tiết theo HTTP 404/410 "
    f"và {unresolved:,} tin chưa có kết quả chắc chắn. "
    "Các con số phản ánh dữ liệu trên đĩa và thời điểm ghi trong nhật ký."
))

**Giới hạn cần giữ khi sử dụng dữ liệu**

- Đây là giá rao bán và thông tin tự khai báo; cần làm sạch, kiểm tra giá trị bất hợp lý trước khi phân tích hoặc định giá.
- Khử trùng theo mã tin không đồng nghĩa đã khử trùng theo chiếc xe. Chia dải giá rồi dừng ở số lượng mục tiêu cũng có thể làm phân bố mẫu khác thị trường.
- Các trường mã phân loại được giữ nguyên để notebook 02 giải mã; giá trị thiếu vẫn giữ nguyên trong dữ liệu thô.
- Kết quả mô tả phản ánh thời điểm cào bổ sung. Notebook 04 cần công bố độ phủ và cách dùng tiêu đề cho phần thiếu mô tả trên đúng tập dữ liệu phân tích.

**Bước tiếp theo:** chuyển `cars.csv` sang notebook 02; chuyển `descriptions.csv` cùng nhật ký và báo cáo sang quy trình phân tích văn bản của notebook 04.